# 07 · Inference, bulk translation and export

**What this notebook is for**

1. A simple `translate()` you can call on any sentence.
2. Bulk-translating the English PSA corpus into Ekegusii — which produces the
   candidate data for human post-editing, the highest-value next step.
3. Saving and optionally publishing the model.

**Runtime** — minutes for spot checks; 1–2 hours for the full 50k corpus.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
sys.path.insert(0, str(pathlib.Path.cwd()))
import nb_common as C

C.set_seed()
C.use_house_style()
print(f"project root: {C.ROOT}")

In [ ]:
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(C.FINETUNED_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    C.FINETUNED_MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()
print(f"loaded {C.FINETUNED_MODEL.name} on {device}")

## 1. Translate anything

In [ ]:
EOS = tok.eos_token_id

@torch.no_grad()
def translate(texts, src_lang=C.ENG, tgt_lang=C.GUZ, batch=32, beams=4, max_len=128):
    if isinstance(texts, str):
        texts = [texts]
    out = []
    tgt_id = tok.convert_tokens_to_ids(tgt_lang)
    pad = tok.pad_token_id
    for i in range(0, len(texts), batch):
        chunk = texts[i:i + batch]
        enc = [[tok.convert_tokens_to_ids(src_lang)] +
               tok(t, add_special_tokens=False, truncation=True,
                   max_length=max_len - 2)["input_ids"] + [EOS] for t in chunk]
        m = max(len(e) for e in enc)
        ids = torch.tensor([[pad] * (m - len(e)) + e for e in enc]).to(device)
        gen = model.generate(input_ids=ids, attention_mask=(ids != pad).long(),
                             forced_bos_token_id=tgt_id,
                             max_new_tokens=max_len, num_beams=beams)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return out

samples = [
    "Report suspected cholera cases to the nearest health facility immediately.",
    "Apply for HELB loans before the September deadline through the online portal.",
    "Vaccinate your livestock against foot and mouth disease before the rains begin.",
    "Avoid crossing flooded rivers and follow county safety guidance.",
]
for lang, name in [(C.GUZ, "Ekegusii"), (C.SWH, "Kiswahili")]:
    print(f"\n--- English -> {name} ---")
    for s, t in zip(samples, translate(samples, C.ENG, lang)):
        print(f"  EN : {s}")
        print(f"  {name[:3].upper()}: {t}\n")

## 2. Bulk-translate the PSA corpus into Ekegusii

This is not a finished dataset — it is **machine output awaiting human
post-editing**. Post-editing a few thousand of these with native speakers gives
you genuine Ekegusii PSA data, which is worth far more than any amount of extra
synthetic text and is the natural second iteration of this project.

In [ ]:
RUN_BULK = False          # set True when you are ready to spend the GPU time
LIMIT    = None           # e.g. 5000 to do a slice first

if RUN_BULK:
    psa = pd.read_csv(C.PSA_PARALLEL_CSV)
    if LIMIT:
        psa = psa.head(LIMIT)
    texts = psa["English"].astype(str).tolist()
    print(f"translating {len(texts):,} PSAs -> Ekegusii ...")
    psa["Ekegusii_mt"] = translate(texts, C.ENG, C.GUZ, batch=64)
    psa["needs_post_editing"] = True
    out = C.OUTPUT / "psa_ekegusii_machine_draft.csv"
    psa.to_csv(out, index=False, encoding="utf-8")
    print(f"wrote {out}")
else:
    print("RUN_BULK is False - set it to True to translate the whole corpus.")

## 3. Post-editing worksheet

Give your native speakers something easy to work with: a stratified sample with
an empty column to correct in, and a rating column so you also learn *how* wrong
the model is.

In [ ]:
SAMPLE_PER_DOMAIN = 60

draft_path = C.OUTPUT / "psa_ekegusii_machine_draft.csv"
if draft_path.exists():
    d = pd.read_csv(draft_path)
    parts = [g.sample(min(SAMPLE_PER_DOMAIN, len(g)), random_state=C.SEED)
             for _, g in d.groupby("Domain")] if "Domain" in d.columns else [d.head(300)]
    sheet = pd.concat(parts)[["PSA_Id", "Domain", "English", "Ekegusii_mt"]] \
        if "PSA_Id" in d.columns else pd.concat(parts)[["English", "Ekegusii_mt"]]
    sheet["Ekegusii_corrected"] = ""
    sheet["adequacy_1_to_4"] = ""
    sheet["register_ok_y_n"] = ""
    sheet["notes"] = ""
    out = C.OUTPUT / "post_editing_worksheet.csv"
    sheet.to_csv(out, index=False, encoding="utf-8")
    print(f"wrote {len(sheet)} rows -> {out}")
    print("\nadequacy: 1 = meaning lost, 4 = fully correct")
    print("register: does it sound like a public notice rather than scripture?")
else:
    print("Run section 2 first to produce the machine draft.")

## 4. Export

`save_pretrained` writes a self-contained model directory. Pushing to the Hugging
Face Hub is optional — but note that the corpus is derived from eBible texts and
community-contributed material, so check the licence terms before publishing
anything.

In [ ]:
EXPORT_DIR = C.ARTIFACTS / "export" / "nllb600m-ekegusii-psa"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(EXPORT_DIR)
tok.save_pretrained(EXPORT_DIR)
print(f"exported -> {EXPORT_DIR}")

PUSH_TO_HUB = False
if PUSH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()
    repo = "your-username/nllb-200-600M-ekegusii-psa"
    model.push_to_hub(repo); tok.push_to_hub(repo)
    print(f"pushed -> https://huggingface.co/{repo}")

## Where to go next

1. **Get the human PSA test set built** (100–300 sentences, stratified across the
   five domains). Until it exists, notebook 06 can only report biblical-domain
   Ekegusii quality, and no claim about PSA translation is supportable.
2. **Post-edit a few thousand machine drafts** from section 2 and retrain. This
   is the fastest route to genuine in-domain Ekegusii data.
3. **Chase civic vocabulary.** Notebook 01 showed that 53.6% of PSA content-word
   types never appear in the training data — institutions, portals, bursaries.
   County government notices, health leaflets and radio scripts in Ekegusii
   would close that gap in a way no amount of scripture can.